In [1]:
from pgmpy.models import LinearGaussianBayesianNetwork
from pgmpy.factors.continuous import LinearGaussianCPD
from sklearn.model_selection import train_test_split
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


In [2]:
# This notebook uses the pre-saved GBNs and their train/val/test datasets from GBN_Data.
# It does not generate new GBNs inside the experiment loop.


In [3]:


def introduce_missingness(df, missing_fraction, seed=None):
    rng = np.random.default_rng(seed)
    df_missing = df.copy()

    # Create a mask of entries to set as missing
    mask = rng.random(df.shape) < missing_fraction

    df_missing[mask] = np.nan
    return df_missing


def split_data(data_frame, train_frac, val_frac, test_frac, seed):
    train_val_data, test_data = train_test_split(data_frame, test_size=test_frac, random_state=seed)
    val_size = val_frac / (train_frac + val_frac)
    train_data, val_data = train_test_split(train_val_data, test_size=val_size, random_state=seed)
    return train_data.reset_index(drop=True), val_data.reset_index(drop=True), test_data.reset_index(drop=True)

def mean_impute(train_df, val_df, test_df):
    """
    Mean imputation fitted on training data only.
    """
    impute_values = train_df.mean()

    train_imp = train_df.fillna(impute_values)
    val_imp   = val_df.fillna(impute_values)
    test_imp  = test_df.fillna(impute_values)

    return train_imp, val_imp, test_imp


In [4]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

MIN_STD = 1e-3
USE_RIDGE_CV = True
RIDGE_ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
RIDGE_CV_FOLDS = 3


def train_baseline(true_gbn, train_data):
    gbn_copy = true_gbn.copy()
    gbn_copy.fit(train_data)
    nodes = list(gbn_copy.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    parents = []

    for node in nodes:
        parent_names = true_gbn.get_parents(node)
        parent_indices = [node_to_idx[p] for p in parent_names]
        parents.append(parent_indices)

    return gbn_copy, parents, nodes


def fit_ridge_for_node(X, y, use_cv=True, alpha_grid=None, cv_folds=3, min_std=MIN_STD):
    if alpha_grid is None:
        alpha_grid = RIDGE_ALPHA_GRID

    if X.shape[1] == 0:
        mu = float(np.mean(y))
        resid = y - mu
        sigma = max(float(np.std(resid, ddof=1)), min_std)
        return mu, np.array([], dtype=float), sigma, np.nan

    if use_cv:
        ridge = Ridge(fit_intercept=True)
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        grid = GridSearchCV(
            estimator=ridge,
            param_grid={"alpha": alpha_grid},
            scoring="neg_mean_squared_error",
            cv=cv,
            n_jobs=-1
        )
        grid.fit(X, y)
        model = grid.best_estimator_
        alpha_used = float(grid.best_params_["alpha"])
    else:
        model = Ridge(alpha=1.0, fit_intercept=True)
        model.fit(X, y)
        alpha_used = float(model.alpha)

    y_hat = model.predict(X)
    resid = y - y_hat
    sigma = max(float(np.std(resid, ddof=1)), min_std)

    return float(model.intercept_), np.array(model.coef_, dtype=float), sigma, alpha_used


def train_ridge_baseline(train_data, nodes, parents_dict, use_cv=True, alpha_grid=None, cv_folds=3, min_std=MIN_STD):
    mu_pred = []
    weights_pred = []
    sigma_pred = []
    alpha_used = []

    for node in nodes:
        parent_names = parents_dict[node]
        y = train_data[node].values.astype(float)
        X = train_data[parent_names].values.astype(float) if len(parent_names) > 0 else np.zeros((len(y), 0))

        mu, w, sigma, alpha = fit_ridge_for_node(
            X=X,
            y=y,
            use_cv=use_cv,
            alpha_grid=alpha_grid,
            cv_folds=cv_folds,
            min_std=min_std
        )

        mu_pred.append(mu)
        weights_pred.append(w)
        sigma_pred.append(sigma)
        alpha_used.append(alpha)

    return mu_pred, weights_pred, sigma_pred, alpha_used


In [5]:
def extract_gbn_parameters(gbn, nodes, min_std=1e-3):
    """
    Extract parents, intercepts, weights, and stds
    from a Linear Gaussian Bayesian Network.
    """

    parents_dict = {}
    mu_true = []
    weights_true = []
    sigma_true = []

    for node in nodes:
        parents = list(gbn.get_parents(node))
        parents_dict[node] = parents

        cpd = gbn.get_cpds(node)

        # Intercept
        mu_true.append(float(cpd.beta[0]))

        # Weights
        if len(cpd.beta) > 1:
            w = np.array(cpd.beta[1:].flatten(), dtype=float)
        else:
            w = np.array([])
        weights_true.append(w)

        # Std (with lower bound)
        sigma_true.append(max(float(cpd.std), min_std))

    return parents_dict, mu_true, weights_true, sigma_true


In [6]:
# =========================================
# Node-wise NN without dropout
# =========================================
class NodeNN(nn.Module):
    def __init__(self, n_parents, hidden_dims, activation):
        super().__init__()

        input_dim = n_parents if n_parents > 0 else 1
        dims = [input_dim] + hidden_dims

        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if activation is not None:
                layers.append(activation())

        self.feature_extractor = nn.Sequential(*layers) if layers else nn.Identity()

        final_dim = dims[-1] if dims else input_dim
        self.mean_out = nn.Linear(final_dim, 1)
        self.logvar_out = nn.Linear(final_dim, 1)

    def forward(self, x):
        x = self.feature_extractor(x)
        mu = self.mean_out(x).squeeze(-1)
        log_var = self.logvar_out(x).squeeze(-1)
        return mu, log_var


# =========================================
# Gaussian NLL
# =========================================
""" def gaussian_nll(mu, log_var, target):
    log_var = torch.clamp(log_var, min=-10.0, max=10.0)  # prevent extreme values
    var = torch.exp(log_var)
    return 0.5 * torch.mean(log_var + (target - mu) ** 2 / var) """


def gaussian_nll(mu, log_var, target):
    log_var = torch.clamp(log_var, min=-10.0)
    var = torch.exp(log_var)
    device = mu.device  # ensure same device as input
    return 0.5 * torch.mean(torch.log(torch.tensor(2 * torch.pi, device=device)) + log_var + ((target - mu)**2)/var)




# =========================================
# Train node-wise NNs
# =========================================
def train_nodewise_nns(
    df_train,
    df_val,
    parents,
    nodes,
    hidden_dims,
    activation,
    device="cpu",
    n_epochs=500,
    lr=1e-3,
    patience=20
):
    nn_models = []
    optimizers = []
    node_train_times = []
    # Create models
    for node in nodes:
        model = NodeNN(
            n_parents=len(parents[node]),
            hidden_dims=hidden_dims,
            activation=activation
        ).to(device)
        nn_models.append(model)
        optimizers.append(torch.optim.Adam(model.parameters(), lr=lr))

    # Early stopping state
    best_val_losses = [float("inf")] * len(nodes)
    patience_counters = [0] * len(nodes)
    best_states = [None] * len(nodes)
    stopped = [False] * len(nodes)

    # Training loop
    for epoch in range(n_epochs):
        epoch_train_loss = 0.0
        active_nodes = 0

        for i, node in enumerate(nodes):
            if stopped[i]:
                continue
            
            start_time = time.time()
            model = nn_models[i]
            optimizer = optimizers[i]
            model.train()

            # TRAIN
            x_train = torch.tensor(df_train[parents[node]].values, dtype=torch.float32, device=device) \
                if parents[node] else torch.zeros((len(df_train), 1), dtype=torch.float32, device=device)
            y_train = torch.tensor(df_train[node].values, dtype=torch.float32, device=device)

            optimizer.zero_grad()
            mu_pred, log_var_pred = model(x_train)
            train_loss = gaussian_nll(mu_pred, log_var_pred, y_train)
            train_loss.backward()
            optimizer.step()

            epoch_train_loss += train_loss.item()
            active_nodes += 1

            end_time = time.time()

            if len(node_train_times) <= i:
                node_train_times.append(end_time - start_time)
            else:
                node_train_times[i] += end_time - start_time

            # VALIDATION
            model.eval()
            with torch.no_grad():
                x_val = torch.tensor(df_val[parents[node]].values, dtype=torch.float32, device=device) \
                    if parents[node] else torch.zeros((len(df_val), 1), dtype=torch.float32, device=device)
                y_val = torch.tensor(df_val[node].values, dtype=torch.float32, device=device)

                mu_val, log_var_val = model(x_val)
                val_loss = gaussian_nll(mu_val, log_var_val, y_val).item()

            # EARLY STOPPING
            if val_loss < best_val_losses[i]:
                best_val_losses[i] = val_loss
                patience_counters[i] = 0
                best_states[i] = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                patience_counters[i] += 1
                if patience_counters[i] >= patience:
                    stopped[i] = True
                    model.load_state_dict(best_states[i])
                    print(f"Node {node} early stopped at epoch {epoch+1}. Best val loss={best_val_losses[i]:.4f}")

        if (epoch + 1) % 50 == 0 and active_nodes > 0:
            print(f"Epoch {epoch+1}, Avg Train Loss: {epoch_train_loss / active_nodes:.4f}")

        if all(stopped):
            print(f"All nodes early-stopped at epoch {epoch+1}.")
            break

    return nn_models, best_val_losses, node_train_times


In [7]:
# ===============================
# 4. KL Divergence Computation
# ===============================
def kl_divergence_gaussian(mu_p, sigma_p, mu_q, sigma_q):
    return 0.5 * (sigma_p**2 / sigma_q**2 + ((mu_q - mu_p)**2) / sigma_q**2 - 1 + np.log(sigma_q**2 / sigma_p**2))


def kl_divergence_gaussian(mu_p, sigma_p, mu_q, sigma_q):
    """KL divergence between two univariate Gaussians."""
    return 0.5 * (
        (sigma_p**2 / sigma_q**2)
        + ((mu_q - mu_p)**2) / sigma_q**2
        - 1
        + np.log(sigma_q**2 / sigma_p**2)
    )

def compute_baseline_lgbn_kl(
    data,
    mu_true,
    weights_true,
    sigma_true,
    mu_pred,
    weights_pred,
    sigma_pred,
    parents
):
    """
    Compute node-wise KL between true and predicted Gaussian CPDs.
    - parents: dict keyed by node names (strings), e.g., "X_0", "X_1"
    - mu_true, weights_true, sigma_true, mu_pred, weights_pred, sigma_pred: lists
    """
    X_values = data.values
    # sort node names to ensure consistent order
    nodes = sorted(parents.keys(), key=lambda x: int(x.split("_")[1]))
    N_nodes = len(nodes)

    # map node names -> list indices
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    kl_node_list = np.zeros(N_nodes)

    for idx, node in enumerate(nodes):
        parent_nodes = parents[node]

        # get index in the lists
        i = idx  # index in mu_true, weights_true, etc.

        if parent_nodes:
            # convert parent names to integer indices in X_values
            parent_idx = [node_to_idx[p] for p in parent_nodes]

            mu_p = mu_true[i] + np.sum(X_values[:, parent_idx] * weights_true[i], axis=1)
            mu_q = mu_pred[i] + np.sum(X_values[:, parent_idx] * weights_pred[i], axis=1)
        else:
            mu_p = mu_true[i]
            mu_q = mu_pred[i]

        var_p = sigma_true[i] ** 2
        var_q = sigma_pred[i] ** 2

        kl = 0.5 * (np.log(var_q / var_p) + (var_p + (mu_p - mu_q) ** 2) / var_q - 1)
        kl_node_list[i] = np.mean(kl)

    return kl_node_list, kl_node_list.mean()


# =========================================
# 5. Compute KL divergence using DataFrame
# =========================================

def nodewise_kl_gaussian(mu_true, sigma_true, mu_pred, log_var_pred, eps=1e-6):
    log_var_pred_clamped = torch.clamp(log_var_pred, min=-10, max=10)
    sigma_pred = torch.exp(0.5 * log_var_pred_clamped) + eps
    sigma_true_safe = torch.clamp(sigma_true, min=eps)
    term1 = torch.log(sigma_pred / sigma_true_safe)
    term2 = (sigma_true_safe**2 + (mu_true - mu_pred)**2) / (2 * sigma_pred**2)
    kl = term1 + term2 - 0.5
    kl_avg = kl.mean().item()
    return kl, kl_avg

def compute_nn_lgbn_kl(df_test, mu_true, weights_true, sigma_true, nn_models, parents, device='cpu'):
    N_samples, N_nodes = df_test.shape
    kl_node = np.zeros(N_nodes)
    nodes = df_test.columns.tolist()

    for i in range(N_nodes):
        if parents[i]:  # node has parents
            parent_indices = [idx for idx in parents[i] if idx < N_nodes]
            if len(parent_indices) == 0:
                mu_p_node = np.full(N_samples, mu_true[i])
            else:
                parent_values = df_test.iloc[:, parent_indices].values
                mu_p_node = mu_true[i] + parent_values @ weights_true[i]
        else:
            mu_p_node = np.full(N_samples, mu_true[i])
        sigma_p_node = sigma_true[i]

        # NN prediction
        if parents[i]:
            x_input = torch.tensor(df_test.iloc[:, parent_indices].values, dtype=torch.float32).to(device)
        else:
            x_input = torch.zeros((N_samples,1), dtype=torch.float32).to(device)

        mu_q_node, log_var_q = nn_models[i](x_input)
        mu_q_node = mu_q_node.detach().cpu().numpy()
        sigma_q_node = torch.exp(0.5 * torch.clamp(log_var_q, -10, 10)).cpu().numpy()


        kl_samples = 0.5 * (sigma_p_node**2 / sigma_q_node**2 + ((mu_q_node - mu_p_node)**2) / sigma_q_node**2 - 1 + np.log(sigma_q_node**2 / sigma_p_node**2))
        kl_node[i] = np.mean(kl_samples)

    kl_avg = np.sum(kl_node)/N_nodes
    return kl_node, kl_avg




In [8]:
def baseline_conditional_gaussian_kl(
    true_mu: torch.Tensor,          # scalar
    true_weights: torch.Tensor,     # (n_parents,)
    true_sigma: torch.Tensor,       # scalar
    base_mu: torch.Tensor,          # scalar
    base_weights: torch.Tensor,     # (n_parents,)
    base_sigma: torch.Tensor,       # scalar
    x: torch.Tensor                 # (N, n_parents)
):
    """
    Computes KL( p(y|x) || q_base(y|x) ) averaged over samples.

    p(y|x)      = N(true_mu + x @ true_weights, true_sigma^2)
    q_base(y|x) = N(base_mu + x @ base_weights, base_sigma^2)
    """

    # Conditional means
    mu_true_cond = true_mu + x @ true_weights
    mu_base_cond = base_mu + x @ base_weights

    # Variances
    var_true = true_sigma ** 2
    var_base = base_sigma ** 2

    # KL per sample
    kl_per_sample = (
        0.5 * (
            torch.log(var_base / var_true)
            + (var_true + (mu_true_cond - mu_base_cond) ** 2) / var_base
            - 1.0
        )
    )

    return kl_per_sample.mean()



def conditional_gaussian_nn_kl(
    true_mu: torch.Tensor,          # scalar
    true_weights: torch.Tensor,     # (n_parents,)
    true_sigma: torch.Tensor,       # scalar
    x: torch.Tensor,                # (N, n_parents)
    pred_mu: torch.Tensor,          # (N,)
    pred_log_var: torch.Tensor      # (N,)
):
    """
    Computes KL( p(y|x) || q(y|x) ) averaged over samples.

    p(y|x) = N(true_mu + x @ true_weights, true_sigma^2)
    q(y|x) = N(pred_mu(x), exp(pred_log_var(x)))
    """

    # True conditional mean
    mu_true_cond = true_mu + x @ true_weights

    # Variances
    var_true = true_sigma ** 2
    var_pred = torch.exp(pred_log_var)

    # KL per sample
    kl_per_sample = (
        0.5 * (
            torch.log(var_pred / var_true)
            + (var_true + (mu_true_cond - pred_mu) ** 2) / var_pred
            - 1.0
        )
    )

    return kl_per_sample.mean()


In [9]:
def save_gbn_parameters(gbn, node_names, save_path):
    """
    Save GBN structure + parameters in JSON-serialisable form.
    """

    parents, mu, weights, sigma = extract_gbn_parameters(gbn, node_names)

    # ---- Make JSON-safe ----
    mu_json = [float(m) for m in mu]
    sigma_json = [float(s) for s in sigma]

    weights_json = []
    for w in weights:
        if isinstance(w, np.ndarray):
            weights_json.append(w.tolist())
        else:
            weights_json.append([])

    gbn_dict = {
        "nodes": list(node_names),
        "parents": parents,
        "mu": mu_json,
        "sigma": sigma_json,
        "weights": weights_json,
        "edges": [(str(u), str(v)) for u, v in gbn.edges()]
    }

    with open(save_path, "w") as f:
        json.dump(gbn_dict, f, indent=2)


def save_ridge_parameters(node_names, parents_dict, mu_pred, weights_pred, sigma_pred, alpha_used, save_path):
    """
    Save ridge baseline parameters in JSON-serialisable form.
    """
    ridge_dict = {
        "nodes": list(node_names),
        "parents": parents_dict,
        "mu": [float(m) for m in mu_pred],
        "sigma": [float(s) for s in sigma_pred],
        "weights": [w.tolist() if isinstance(w, np.ndarray) else list(w) for w in weights_pred],
        "alpha_used": [None if pd.isna(a) else float(a) for a in alpha_used],
    }

    with open(save_path, "w") as f:
        json.dump(ridge_dict, f, indent=2)


import json
import torch
import numpy as np


def save_nn_predictions(nn_models, node_names, parents_dict, test_data, node_train_times, save_path, device="cpu", hidden_dims=None, activation=None):
    """
    Save node-wise NN predicted mu, sigma (exp(log_var)), and training time on test_data.
    """
    mu_pred_list = []
    sigma_pred_list = []

    for idx, node in enumerate(node_names):
        model = nn_models[idx]
        model.eval()
        if parents_dict[node]:
            x_input = torch.tensor(test_data[parents_dict[node]].values, dtype=torch.float32, device=device)
        else:
            x_input = torch.zeros((len(test_data), 1), dtype=torch.float32, device=device)

        with torch.no_grad():
            mu_pred, log_var_pred = model(x_input)
            mu_pred_list.append(mu_pred.cpu().numpy().tolist())
            sigma_pred_list.append(torch.exp(log_var_pred).cpu().numpy().tolist())

    nn_dict = {
        "nodes": list(node_names),
        "parents": parents_dict,
        "mu_pred": mu_pred_list,
        "sigma_pred": sigma_pred_list,
        "train_time_sec": node_train_times
    }

    if hidden_dims is not None:
        nn_dict["hidden_dims"] = hidden_dims
    if activation is not None:
        nn_dict["activation_func"] = "linear" if activation is None else activation.__name__

    with open(save_path, "w") as f:
        json.dump(nn_dict, f, indent=2)


In [10]:
def load_dataset(base_dir, gbn_size, max_indegree, sim, n_samples):
    base_dir = Path(base_dir)

    data_dir = (
        base_dir
        / f"GBN_{gbn_size}"
        / f"GBN_{gbn_size}_indegree_{max_indegree}"
        / f"{sim}. GBN_{gbn_size}_indegree_{max_indegree}"
        / f"GBN_{gbn_size}_indegree_{max_indegree}_samples_{n_samples}"
    )

    train_df = pd.read_csv(data_dir / "train.csv")
    val_df   = pd.read_csv(data_dir / "val.csv")
    test_df  = pd.read_csv(data_dir / "test.csv")

    return train_df, val_df, test_df



def load_true_gbn(base_dir, gbn_size, max_indegree, sim):
    base_dir = Path(base_dir)

    gbn_path = (
        base_dir
        / f"GBN_{gbn_size}"
        / f"GBN_{gbn_size}_indegree_{max_indegree}"
        / f"{sim}. GBN_{gbn_size}_indegree_{max_indegree}"
        / "true_gbn.json"
    )

    with open(gbn_path, "r") as f:
        gbn_dict = json.load(f)

    return gbn_dict




def reconstruct_gbn_from_json(gbn_dict):
    """
    Reconstructs a Linear Gaussian Bayesian Network from JSON saved by save_gbn_parameters.

    Args:
        gbn_dict (dict): JSON dictionary containing:
            - 'nodes': list of node names
            - 'parents': dict {node: list of parent nodes}
            - 'mu': list of intercepts
            - 'sigma': list of std devs
            - 'weights': list of weights per node (list of lists)

    Returns:
        LinearGaussianBayesianNetwork: pgmpy model with CPDs attached
    """

    node_names = sorted(gbn_dict["nodes"], key=lambda x: int(x.split("_")[1]))
    parents_dict = gbn_dict["parents"]
    mu_list = gbn_dict["mu"]
    sigma_list = gbn_dict["sigma"]
    weights_list = gbn_dict["weights"]

    # Build the DAG structure
    edges = [(p, node) for node, parents in parents_dict.items() for p in parents]
    model = LinearGaussianBayesianNetwork(edges)

    cpds = []

    for idx, node in enumerate(node_names):
        parents = parents_dict[node]
        mu = mu_list[idx]
        sigma = sigma_list[idx]
        weights = weights_list[idx]

        # beta = [intercept, w1, w2, ...]
        beta = [mu] + weights if weights else [mu]

        # Create CPD
        if parents:
            cpd = LinearGaussianCPD(node, beta, sigma, evidence=parents)
        else:
            cpd = LinearGaussianCPD(node, beta, sigma)

        cpds.append(cpd)

    # Add all CPDs at once
    model.add_cpds(*cpds)

    # Verify model
    if not model.check_model():
        raise ValueError("GBN reconstruction failed!")

    return model




In [12]:
from pathlib import Path

# -------------------------
# Experiment setup
# -------------------------
experiment_configs = [
    (10, 2),
    (10, 5),
    (20, 2),
    (20, 5),
    (20, 10),
    (50, 2),
    (50, 5),
    (50, 10),
]
sim = 5
sample_size = [1400]

hidden_dims = [
    [32, 32, 32],
    [32, 16, 32],
    [64, 64, 64],
]
act_func = [nn.ReLU]
missing_frac = [0.1, 0.2, 0.3]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Ridge CV enabled: {USE_RIDGE_CV} | alpha grid: {RIDGE_ALPHA_GRID}")

RAW_DATA_DIR = Path("GBN_Data")
EXPERIMENT_RESULTS_DIR = Path("2Experiment_Results")
EXPERIMENT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Main loop
# -------------------------
records = []
training_time_records = []

for n_nodes_curr, in_deg in experiment_configs:

    for i_gbn in range(1, sim + 1):
        print(f"===== GBN {i_gbn} | Nodes={n_nodes_curr} | In-degree={in_deg} =====")

        sim_dir = RAW_DATA_DIR / f"GBN_{n_nodes_curr}" / f"GBN_{n_nodes_curr}_indegree_{in_deg}" / f"{i_gbn}. GBN_{n_nodes_curr}_indegree_{in_deg}"

        with open(os.path.join(sim_dir, "true_gbn.json"), "r") as f:
            gbn_dict = json.load(f)

        node_names = sorted(gbn_dict["nodes"], key=lambda x: int(x.split("_")[1]))
        true_gbn = reconstruct_gbn_from_json(gbn_dict)
        true_parents_dict, true_mu, true_weights, true_sigma = extract_gbn_parameters(true_gbn, node_names)

        for samples in sample_size:
            print(f"================================================ Number of Samples {samples} ================================================")

            raw_sample_dir = sim_dir / f"{i_gbn}. GBN_{n_nodes_curr}_indegree_{in_deg}_samples_{samples}"

            train_full = pd.read_csv(os.path.join(raw_sample_dir, "train.csv"))
            val_full = pd.read_csv(os.path.join(raw_sample_dir, "val.csv"))
            test_full = pd.read_csv(os.path.join(raw_sample_dir, "test.csv"))

            results_sample_dir = os.path.join(
                EXPERIMENT_RESULTS_DIR,
                f"GBN_{n_nodes_curr}",
                f"GBN_{n_nodes_curr}_indegree_{in_deg}",
                str(i_gbn),
                f"{i_gbn}.GBN_{n_nodes_curr}_indegree_{in_deg}_samples_{samples}"
            )
            os.makedirs(results_sample_dir, exist_ok=True)

            for miss_frac in missing_frac:
                print(f"================================================ Missing Fraction {miss_frac} ================================================")

                exp_dir = os.path.join(results_sample_dir, f"Missing_{miss_frac}")
                os.makedirs(exp_dir, exist_ok=True)

                seed = hash((n_nodes_curr, in_deg, i_gbn, samples, miss_frac)) % 10000

                train_incomplete = introduce_missingness(train_full, miss_frac, seed=seed)
                val_incomplete = introduce_missingness(val_full, miss_frac, seed=seed + 1)
                test_incomplete = introduce_missingness(test_full, miss_frac, seed=seed + 2)

                train_incomplete.to_csv(os.path.join(exp_dir, "incomplete_train.csv"), index=False)
                val_incomplete.to_csv(os.path.join(exp_dir, "incomplete_val.csv"), index=False)
                test_incomplete.to_csv(os.path.join(exp_dir, "incomplete_test.csv"), index=False)

                train_data, val_data, test_data = mean_impute(train_incomplete, val_incomplete, test_incomplete)

                train_data.to_csv(os.path.join(exp_dir, "imputed_train.csv"), index=False)
                val_data.to_csv(os.path.join(exp_dir, "imputed_val.csv"), index=False)
                test_data.to_csv(os.path.join(exp_dir, "imputed_test.csv"), index=False)

                # -----------------------------
                # Train pgmpy baseline and record time
                # -----------------------------
                baseline_start = time.perf_counter()
                baseline_gbn, parents_list, baseline_nodes = train_baseline(true_gbn, train_data)
                baseline_train_time_sec = float(time.perf_counter() - baseline_start)

                save_gbn_parameters(
                    baseline_gbn,
                    sorted(baseline_nodes, key=lambda x: int(x.split("_")[1])),
                    os.path.join(exp_dir, "baseline_gbn.json")
                )

                baseline_parents_dict, baseline_mu, baseline_weights, baseline_sigma = extract_gbn_parameters(
                    baseline_gbn, node_names
                )

                kl_base_node_list = []
                for idx, node in enumerate(node_names):
                    if true_parents_dict[node]:
                        parent_cols = true_parents_dict[node]
                        x_input = torch.tensor(test_data[parent_cols].values, dtype=torch.float32, device=device)
                        w_t = torch.tensor(true_weights[idx], dtype=torch.float32, device=device)
                        w_b = torch.tensor(baseline_weights[idx], dtype=torch.float32, device=device)
                    else:
                        x_input = torch.zeros((len(test_data), 1), dtype=torch.float32, device=device)
                        w_t = torch.zeros(1, dtype=torch.float32, device=device)
                        w_b = torch.zeros(1, dtype=torch.float32, device=device)

                    mu_t = torch.tensor(true_mu[idx], dtype=torch.float32, device=device)
                    sigma_t = torch.tensor(true_sigma[idx], dtype=torch.float32, device=device)
                    mu_b = torch.tensor(baseline_mu[idx], dtype=torch.float32, device=device)
                    sigma_b = torch.tensor(baseline_sigma[idx], dtype=torch.float32, device=device)

                    kl_node = baseline_conditional_gaussian_kl(mu_t, w_t, sigma_t, mu_b, w_b, sigma_b, x_input)
                    kl_base_node_list.append(kl_node.item())

                kl_baseline_avg = sum(kl_base_node_list) / len(kl_base_node_list)

                # -----------------------------
                # Train ridge baseline and record time
                # -----------------------------
                ridge_start = time.perf_counter()
                ridge_mu, ridge_weights, ridge_sigma, ridge_alpha_used = train_ridge_baseline(
                    train_data=train_data[node_names].copy(),
                    nodes=node_names,
                    parents_dict=true_parents_dict,
                    use_cv=USE_RIDGE_CV,
                    alpha_grid=RIDGE_ALPHA_GRID,
                    cv_folds=RIDGE_CV_FOLDS,
                    min_std=MIN_STD
                )
                ridge_train_time_sec = float(time.perf_counter() - ridge_start)

                save_ridge_parameters(
                    node_names=node_names,
                    parents_dict=true_parents_dict,
                    mu_pred=ridge_mu,
                    weights_pred=ridge_weights,
                    sigma_pred=ridge_sigma,
                    alpha_used=ridge_alpha_used,
                    save_path=os.path.join(exp_dir, "ridge_gbn.json")
                )

                kl_ridge_node_list = []
                for idx, node in enumerate(node_names):
                    if true_parents_dict[node]:
                        parent_cols = true_parents_dict[node]
                        x_input = torch.tensor(test_data[parent_cols].values, dtype=torch.float32, device=device)
                        w_t = torch.tensor(true_weights[idx], dtype=torch.float32, device=device)
                        w_r = torch.tensor(ridge_weights[idx], dtype=torch.float32, device=device)
                    else:
                        x_input = torch.zeros((len(test_data), 1), dtype=torch.float32, device=device)
                        w_t = torch.zeros(1, dtype=torch.float32, device=device)
                        w_r = torch.zeros(1, dtype=torch.float32, device=device)

                    mu_t = torch.tensor(true_mu[idx], dtype=torch.float32, device=device)
                    sigma_t = torch.tensor(true_sigma[idx], dtype=torch.float32, device=device)
                    mu_r = torch.tensor(ridge_mu[idx], dtype=torch.float32, device=device)
                    sigma_r = torch.tensor(ridge_sigma[idx], dtype=torch.float32, device=device)

                    kl_node = baseline_conditional_gaussian_kl(mu_t, w_t, sigma_t, mu_r, w_r, sigma_r, x_input)
                    kl_ridge_node_list.append(kl_node.item())

                kl_ridge_avg = sum(kl_ridge_node_list) / len(kl_ridge_node_list)
                ridge_alpha_nonroot = [a for a in ridge_alpha_used if not pd.isna(a)]

                # -----------------------------
                # Train NNs and record time per architecture
                # -----------------------------
                exp_results = []

                for dim in hidden_dims:
                    print(f"========================================================== Hidden Dim {dim} ==========================================================")
                    for activation in act_func:
                        activation_name = "linear" if activation is None else activation.__name__
                        print(f"========================================================== Activation Function {activation_name} ==========================================================")

                        nn_start = time.perf_counter()
                        nn_models, best_val_losses, node_train_times = train_nodewise_nns(
                            train_data,
                            val_data,
                            true_parents_dict,
                            node_names,
                            dim,
                            activation,
                            device=device,
                            n_epochs=500,
                            lr=1e-3,
                            patience=20
                        )
                        nn_train_time_sec = float(time.perf_counter() - nn_start)

                        nn_json_path = os.path.join(exp_dir, f"nn_pred_{dim}_{activation_name}.json")
                        save_nn_predictions(
                            nn_models,
                            node_names,
                            true_parents_dict,
                            test_data,
                            node_train_times,
                            nn_json_path,
                            device=device,
                            hidden_dims=dim,
                            activation=activation
                        )

                        kl_nn_node_list = []
                        for idx, model in enumerate(nn_models):
                            node_curr = node_names[idx]
                            model.eval()
                            if true_parents_dict[node_curr]:
                                x_input = torch.tensor(
                                    test_data[true_parents_dict[node_curr]].values,
                                    dtype=torch.float32,
                                    device=device
                                )
                                w_t = torch.tensor(true_weights[idx], dtype=torch.float32, device=device)
                            else:
                                x_input = torch.zeros((len(test_data), 1), dtype=torch.float32, device=device)
                                w_t = torch.zeros(1, dtype=torch.float32, device=device)

                            mu_t = torch.tensor(true_mu[idx], dtype=torch.float32, device=device)
                            sigma_t = torch.tensor(true_sigma[idx], dtype=torch.float32, device=device)

                            with torch.no_grad():
                                mu_pred, log_var_pred = model(x_input)
                                mu_pred = mu_pred.squeeze()
                                log_var_pred = log_var_pred.squeeze()

                            kl_node = conditional_gaussian_nn_kl(mu_t, w_t, sigma_t, x_input, mu_pred, log_var_pred)
                            kl_nn_node_list.append(kl_node.item())

                        kl_nn_avg = sum(kl_nn_node_list) / len(kl_nn_node_list)

                        record = {
                            "n_nodes": n_nodes_curr,
                            "max_in_degree": in_deg,
                            "sim": i_gbn,
                            "n_samples": samples,
                            "missing_frac": miss_frac,
                            "hidden_dims": dim,
                            "activation_func": activation_name,
                            "baseline_kl_avg": kl_baseline_avg,
                            "ridge_kl_avg": kl_ridge_avg,
                            "nn_kl_avg": kl_nn_avg,
                            "nn_val_nll_mean": float(np.mean(best_val_losses)),
                            "nn_val_nll_std": float(np.std(best_val_losses)),
                            "baseline_train_time_sec": baseline_train_time_sec,
                            "ridge_train_time_sec": ridge_train_time_sec,
                            "nn_train_time_sec": nn_train_time_sec,
                            "nn_train_time_min": nn_train_time_sec / 60.0,
                            "nn_train_time_mean_node_sec": float(np.mean(node_train_times)),
                            "nn_train_time_total_node_sec": float(np.sum(node_train_times)),
                            "ridge_alpha_mean_nonroot": float(np.mean(ridge_alpha_nonroot)) if len(ridge_alpha_nonroot) > 0 else np.nan,
                            "ridge_alpha_min_nonroot": float(np.min(ridge_alpha_nonroot)) if len(ridge_alpha_nonroot) > 0 else np.nan,
                            "ridge_alpha_max_nonroot": float(np.max(ridge_alpha_nonroot)) if len(ridge_alpha_nonroot) > 0 else np.nan,
                        }
                        exp_results.append(record)
                        records.append(record)

                        training_time_records.append({
                            "n_nodes": n_nodes_curr,
                            "max_in_degree": in_deg,
                            "sim": i_gbn,
                            "n_samples": samples,
                            "missing_frac": miss_frac,
                            "method": "nn",
                            "hidden_dims": str(dim),
                            "activation_func": activation_name,
                            "train_time_sec": nn_train_time_sec,
                            "train_time_min": nn_train_time_sec / 60.0,
                        })

                training_time_records.append({
                    "n_nodes": n_nodes_curr,
                    "max_in_degree": in_deg,
                    "sim": i_gbn,
                    "n_samples": samples,
                    "missing_frac": miss_frac,
                    "method": "baseline",
                    "hidden_dims": None,
                    "activation_func": None,
                    "train_time_sec": baseline_train_time_sec,
                    "train_time_min": baseline_train_time_sec / 60.0,
                })
                training_time_records.append({
                    "n_nodes": n_nodes_curr,
                    "max_in_degree": in_deg,
                    "sim": i_gbn,
                    "n_samples": samples,
                    "missing_frac": miss_frac,
                    "method": "ridge",
                    "hidden_dims": None,
                    "activation_func": None,
                    "train_time_sec": ridge_train_time_sec,
                    "train_time_min": ridge_train_time_sec / 60.0,
                })

                results_df = pd.DataFrame(exp_results)
                results_df.to_csv(os.path.join(exp_dir, "results.csv"), index=False)

                time_df = pd.DataFrame([
                    {
                        "n_nodes": n_nodes_curr,
                        "max_in_degree": in_deg,
                        "sim": i_gbn,
                        "n_samples": samples,
                        "missing_frac": miss_frac,
                        "method": "baseline",
                        "hidden_dims": None,
                        "activation_func": None,
                        "train_time_sec": baseline_train_time_sec,
                        "train_time_min": baseline_train_time_sec / 60.0,
                    },
                    {
                        "n_nodes": n_nodes_curr,
                        "max_in_degree": in_deg,
                        "sim": i_gbn,
                        "n_samples": samples,
                        "missing_frac": miss_frac,
                        "method": "ridge",
                        "hidden_dims": None,
                        "activation_func": None,
                        "train_time_sec": ridge_train_time_sec,
                        "train_time_min": ridge_train_time_sec / 60.0,
                    },
                    *[
                        {
                            "n_nodes": row["n_nodes"],
                            "max_in_degree": row["max_in_degree"],
                            "sim": row["sim"],
                            "n_samples": row["n_samples"],
                            "missing_frac": row["missing_frac"],
                            "method": "nn",
                            "hidden_dims": str(row["hidden_dims"]),
                            "activation_func": row["activation_func"],
                            "train_time_sec": row["nn_train_time_sec"],
                            "train_time_min": row["nn_train_time_min"],
                        }
                        for row in exp_results
                    ]
                ])
                time_df.to_csv(os.path.join(exp_dir, "method_training_times.csv"), index=False)

all_results_df = pd.DataFrame(records)
all_results_df.to_csv(EXPERIMENT_RESULTS_DIR / "experiment_2_all_results.csv", index=False)

all_training_times_df = pd.DataFrame(training_time_records)
all_training_times_df.to_csv(EXPERIMENT_RESULTS_DIR / "experiment_2_all_training_times.csv", index=False)

print("Saved experiment-level results to:", EXPERIMENT_RESULTS_DIR / "experiment_2_all_results.csv")
print("Saved training-time results to:", EXPERIMENT_RESULTS_DIR / "experiment_2_all_training_times.csv")


Using device: cuda
Ridge CV enabled: True | alpha grid: [0.01, 0.1, 1.0, 10.0, 100.0]
===== GBN 1 | Nodes=20 | In-degree=2 =====
================================================ Number of Samples 1400 ================================================
================================================ Missing Fraction 0.1 ================================================
========================================================== Hidden Dim [32, 32, 32] ==========================================================
========================================================== Activation Function ReLU ==========================================================
Epoch 50, Avg Train Loss: 2.1150
Node X_3 early stopped at epoch 92. Best val loss=1.3758
Epoch 100, Avg Train Loss: 1.7509
Node X_0 early stopped at epoch 123. Best val loss=1.2709
Node X_10 early stopped at epoch 126. Best val loss=0.6111
Node X_14 early stopped at epoch 127. Best val loss=1.5077
Node X_11 early stopped at epoch 129. Best val

c:\Users\ajayn\anaconda3\envs\torch-gpu\lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


========================================================== Hidden Dim [32, 32, 32] ==========================================================
========================================================== Activation Function ReLU ==========================================================
Epoch 50, Avg Train Loss: 4.1577
Node X_14 early stopped at epoch 62. Best val loss=1.5697
Node X_7 early stopped at epoch 68. Best val loss=4.0604
Node X_49 early stopped at epoch 85. Best val loss=1.5298
Node X_15 early stopped at epoch 91. Best val loss=1.5019
Epoch 100, Avg Train Loss: 3.0959
Node X_47 early stopped at epoch 109. Best val loss=1.2839
Node X_38 early stopped at epoch 111. Best val loss=1.3522
Node X_33 early stopped at epoch 121. Best val loss=1.7581
Node X_10 early stopped at epoch 130. Best val loss=0.8677
Node X_48 early stopped at epoch 140. Best val loss=0.9779
Node X_2 early stopped at epoch 143. Best val loss=0.1112
Node X_9 early stopped at epoch 150. Best val loss=0.0677
Epoch 